# Laboratorio 3

## Repositorio

[Link al Repositorio](#)

Una empresa de desarrollo de videojuegos está evaluando el uso de agentes de RL para generar comportamiento no determinista en personajes de un juego de rol táctico. El entorno de prueba es un mapa de cuadrícula de $6 \times 6$ con zonas de recompensa, zonas de penalización y un estado terminal. El equipo técnico necesita entender si Monte Carlo es viable para este dominio antes de escalar a métodos más complejos, y quiere comparar su comportamiento contra la línea base de Programación Dinámica de la semana anterior.

Su grupo ha sido contratado para diseñar el experimento, implementar los métodos, analizar los resultados, y producir un reporte técnico con recomendaciones sobre la viabilidad de Monte Carlo para este dominio.

# Task 1 (Entrega Parcial)

Antes de implementar nada, diseñen formalmente el experimento que van a ejecutar.

## Inciso 1

Especificación del MDP: definan el espacio de estados, el espacio de acciones, la función de recompensa y el factor de descuento $\gamma$ para el mapa $6 \times 6$. Justifiquen cada decisión de diseño. El mapa debe incluir al menos dos zonas de recompensa positiva, una zona de penalización, y un estado terminal. La función de recompensa debe capturar al menos dos objetivos potencialmente conflictivos propios del dominio de videojuegos.

**Respuesta:**

El mapa es una cuadrícula de 6 filas por 6 columnas. Se usan coordenadas $(f, c)$ con $f, c \in \{0, \dots, 5\}$, donde $f$ crece hacia abajo y $c$ hacia la derecha.

| | c=0 | c=1 | c=2 | c=3 | c=4 | c=5 |
|---|---|---|---|---|---|---|
| **f=0** | S | . | . | . | . | . |
| **f=1** | . | . | . | . | T1 | . |
| **f=2** | . | T2 | . | . | . | . |
| **f=3** | . | . | . | P | . | . |
| **f=4** | . | . | . | . | . | . |
| **f=5** | . | . | . | . | . | G |

| Símbolo | Celda | Significado |
|---|---|---|
| S | (0, 0) | Punto de aparición del personaje |
| T1 | (1, 4) | Zona de recompensa (cofre grande) |
| T2 | (2, 1) | Zona de recompensa (cofre pequeño) |
| P | (3, 3) | Zona de penalización (trampa o zona patrullada) |
| G | (5, 5) | Estado terminal (salida de la misión) |

**Espacio de estados**

$$s = (f, c, b_1, b_2), \quad f, c \in \{0, \dots, 5\}, \quad b_1, b_2 \in \{0, 1\}$$

donde $b_1$ y $b_2$ indican si el cofre T1 y el cofre T2 ya fueron recogidos en el episodio actual. Cardinalidad:

$$|S| = 36 \cdot 2 \cdot 2 = 144$$

Los dos bits no son decoración. Sin ellos el personaje puede salir de la celda del cofre y volver a entrar para cobrar la recompensa otra vez, y como el cofre da más de lo que cuestan los dos pasos del ciclo, la política óptima consistiría en dar vueltas sobre el cofre para siempre y nunca llegar a la salida. El episodio dejaría de terminar y la tarea dejaría de ser episódica, que es justo lo que Monte Carlo necesita. Con los bits, el cofre se cobra una sola vez por episodio, tal como funciona un cofre en un juego real. El estado sigue siendo Markov porque el bit resume todo lo que del pasado importa para decidir.

El estado inicial es $s_0 = (0, 0, 0, 0)$ y el estado terminal es cualquier estado con $(f, c) = (5, 5)$, que es absorbente. El episodio también se corta a los 200 pasos para que una política aleatoria no genere trayectorias infinitas.

**Espacio de acciones**

$$A = \{\text{arriba}, \text{abajo}, \text{izquierda}, \text{derecha}\}, \quad |A| = 4$$

Discreto porque el personaje se mueve por celdas, que es como se mueve una unidad en un juego de rol táctico. Las cuatro acciones están disponibles en todo estado. Si el movimiento sale del mapa, el personaje se queda donde está y el paso se consume igual. Esto evita tener que definir $A(s)$ por estado y además le enseña al agente que chocar con el borde desperdicia un turno.

Las transiciones son deterministas: la acción elegida siempre mueve al personaje a la celda correspondiente. El comportamiento no determinista que pide el estudio no viene del entorno sino de la política $\varepsilon$-soft del Inciso 3, que es la que hace que dos partidas no se vean iguales.

**Función de recompensa**

| Evento | Valor |
|---|---|
| Cada paso | −1 |
| Entrar a T1 con $b_1 = 0$ | +5 |
| Entrar a T2 con $b_2 = 0$ | +3 |
| Entrar a P | −10 |
| Llegar a G | +20 |

Las recompensas se suman cuando coinciden en el mismo paso. Por ejemplo, entrar a T1 por primera vez da $-1 + 5 = +4$.

**Objetivos en conflicto**

El diseño pone en tensión dos cosas que cualquier jugador reconoce: terminar la misión rápido y llevarse el botín. El costo de −1 por paso empuja hacia la salida y las zonas de recompensa empujan hacia el desvío. La trampa agrega un tercer eje, porque está sentada sobre la diagonal natural del mapa y la ruta corta tiene que rodearla.

Que el conflicto sea real se ve comparando el retorno de las cuatro rutas razonables, todas esquivando la trampa, con $\gamma = 0.95$:

| Ruta | Pasos | Retorno $G_0$ |
|---|---|---|
| Directa a G | 10 | 4.58 |
| Recoge T1 y sigue a G | 10 | 8.65 |
| Recoge T2 y sigue a G | 10 | 7.29 |
| Recoge T2, luego T1, y termina en G | 12 | 8.57 |

Cada cofre por separado queda sobre una ruta que avanza siempre hacia la salida, así que recoger uno de los dos no cuesta ningún paso extra y siempre conviene. Recoger los dos sí cuesta: obliga a un desvío de 2 pasos y retrasa el +20. Con $\gamma = 0.95$ la mejor ruta es quedarse solo con T1 (8.65) y la segunda mejor es recoger ambos (8.57), una diferencia de 0.09. Es un empate casi perfecto, y es a propósito: sirve para ver si Monte Carlo, con su ruido de muestreo, logra distinguir dos rutas tan cercanas cuando Value Iteration las separa sin problema.

**Factor de descuento**

Se propone $\gamma = 0.95$. El horizonte efectivo es $1/(1-\gamma) = 20$ pasos, y la ruta útil más larga del mapa es de 12 pasos, así que desde cualquier celda el agente todavía alcanza a ver el valor de la salida y no se vuelve miope. Además $\gamma$ es la perilla que decide el estilo del personaje:

| $\gamma$ | Mejor ruta | Personaje que resulta |
|---|---|---|
| 0.90 | Solo T1 (4.52 contra 4.19) | Va a lo suyo y desprecia el desvío |
| 0.95 | Solo T1 (8.65 contra 8.57) | Casi indiferente entre correr y saquear |
| 0.99 | Ambos cofres (14.19 contra 13.51) | Saqueador, el desvío deja de importarle |

La tarea es episódica y termina, así que $\gamma < 1$ no hace falta para que el retorno sea finito. Aquí $\gamma$ se usa como parámetro de diseño de comportamiento, no como truco de convergencia, y se elige 0.95 porque es el punto donde las dos conductas quedan casi empatadas, que es el escenario más exigente para comparar los dos métodos.

## Inciso 2

Selección de variante Monte Carlo: argumenten cuál variante usarán, First-Visit o Every-Visit, y por qué es más apropiada para este dominio específico. Incluyan en su argumento una discusión sobre la frecuencia esperada de revisitas a estados en un mapa $6 \times 6$ bajo una política aleatoria.

**Respuesta:**

Se usará First-Visit Monte Carlo.

**Frecuencia esperada de revisitas**

Bajo política uniforme aleatoria el personaje ejecuta una caminata aleatoria sobre una cuadrícula de 36 celdas con bordes reflejantes. Una caminata aleatoria en 2D es recurrente y no tiene ninguna preferencia por la salida, así que el tiempo esperado para llegar de $(0,0)$ a $(5,5)$ es del orden de cientos de pasos, no de decenas. Con episodios de unos 200 a 300 pasos repartidos entre 36 celdas, cada celda se visita en promedio del orden de 6 a 8 veces por episodio, y las celdas cercanas al punto de aparición bastante más porque la caminata tarda en alejarse. Las revisitas no son un caso raro en este mapa, son la norma.

**Por qué eso favorece a First-Visit**

Every-Visit aprovecharía esas 6 u 8 visitas como 6 u 8 muestras, y a primera vista parece que multiplica los datos. El problema es que los retornos de varias visitas al mismo estado dentro del mismo episodio comparten la misma cola de recompensas: todos terminan con el mismo +20, todos pasan por los mismos cofres, todos cargan la misma trampa si el personaje cayó en ella. Son muestras fuertemente correlacionadas, así que $n$ visitas valen mucho menos que $n$ muestras independientes y la varianza baja mucho menos de lo que sugiere el conteo. Peor aún, esa correlación hace que la barra de error calculada con la desviación muestral quede optimista, y como el objetivo del laboratorio es comparar contra Value Iteration, una barra de error mentirosa arruina la comparación.

First-Visit toma un solo retorno por estado y por episodio. Como los episodios son independientes entre sí, esos retornos sí son independientes e idénticamente distribuidos, el promedio es un estimador insesgado de $V^\pi(s)$ y su varianza cae limpiamente como $\sigma^2/n$ con $n$ igual al número de episodios en los que apareció el estado. Eso permite reportar intervalos de confianza válidos y decir con honestidad cuántos episodios hacen falta para igualar la precisión de Value Iteration.

El costo es que se descartan datos. Se acepta porque en un mapa de 36 celdas con transiciones deterministas simular un episodio es baratísimo: es más fácil correr más episodios que exprimir cada uno. Every-Visit sería la opción razonable si simular fuera caro, por ejemplo si cada episodio requiriera correr el motor del juego completo.

## Inciso 3

Estrategia de exploración: argumenten si usarán Exploring Starts o política $\varepsilon$-soft. Justifiquen su elección considerando si en el dominio de videojuegos es razonable controlar el estado inicial del agente. Propongan un valor concreto de $\varepsilon$ si eligen política $\varepsilon$-soft, con justificación.

**Respuesta:**

Se usará política $\varepsilon$-soft con $\varepsilon = 0.10$.

**Por qué no Exploring Starts**

Exploring Starts exige poder arrancar el episodio en cualquier par $(s, a)$ con probabilidad positiva, y en un videojuego eso choca con el dominio por tres razones concretas:

1. Los personajes aparecen en puntos de aparición diseñados por el equipo de nivel, no en una celda cualquiera. Teletransportar al personaje a una celda arbitraria y forzarle la primera acción rompe la ambientación y no es algo que se pueda hacer en producción, solo en un simulador de laboratorio.
2. En nuestro diseño hay estados que no tienen sentido como inicio. Un estado $(0, 0, 1, 0)$ dice que el personaje está en la entrada pero ya recogió el cofre de $(1, 4)$, algo imposible de alcanzar desde la historia del episodio. Exploring Starts obligaría a inicializar estados inconsistentes con el mundo.
3. Si mañana el estudio quiere que el agente siga aprendiendo mientras la gente juega, no hay forma de reiniciar la partida en un estado arbitrario. $\varepsilon$-soft funciona igual en simulador que en producción.

Además $\varepsilon$-soft encaja con lo que el estudio pidió: comportamiento no determinista. Un agente $\varepsilon$-soft entrenado conserva algo de aleatoriedad después del entrenamiento, así que dos partidas no se ven idénticas sin necesidad de meter ruido artificial encima de la política.

**Por qué $\varepsilon = 0.10$**

Con $|A| = 4$, la política reparte así:

$$\pi(a \mid s) = \begin{cases} 1 - \varepsilon + \dfrac{\varepsilon}{|A|} = 0.925 & \text{acción codiciosa} \\[6pt] \dfrac{\varepsilon}{|A|} = 0.025 & \text{cada una de las otras tres} \end{cases}$$

La ruta óptima tiene 10 pasos, así que la probabilidad de recorrerla completa sin desviarse es $0.925^{10} \approx 0.46$. Cerca de la mitad de los episodios sigue la ruta buena y da datos precisos sobre ella, y la otra mitad se desvía y mantiene actualizados los pares $(s, a)$ alternativos. Es el punto donde ninguna de las dos cosas se muere.

Para ver que el valor no es arbitrario, con $\varepsilon = 0.30$ esa probabilidad cae a $0.775^{10} \approx 0.08$: solo 8 de cada 100 episodios recorren la ruta buena y la estimación de su valor queda dominada por ruido. Con $\varepsilon = 0.01$ pasa lo contrario, las celdas alejadas de la ruta buena casi nunca se visitan y sus valores nunca se aprenden.

Una advertencia sobre las garantías: Monte Carlo con $\varepsilon$-soft converge a la mejor política $\varepsilon$-soft, no a la política óptima. Con $\varepsilon = 0.10$ esa brecha es pequeña, porque en cada paso el personaje se desvía con probabilidad 0.075, pero existe y hay que tomarla en cuenta al comparar contra Value Iteration, que sí devuelve el óptimo exacto. Una forma de cerrarla en el experimento es correr también una variante con $\varepsilon$ decreciente, por ejemplo de 0.30 a 0.05 a lo largo del entrenamiento, y comparar las dos contra la línea base.

## Inciso 4

Hipótesis de comparación: antes de ejecutar el experimento, formulen al menos dos hipótesis concretas sobre cómo esperan que se comporte Monte Carlo en comparación con Value Iteration de la semana pasada. Las hipótesis deben ser falsables y cuantificables.

**Respuesta:**

**Hipótesis 1: coincidencia de políticas y su relación con las visitas**

Después de 50000 episodios de First-Visit Monte Carlo con $\varepsilon = 0.10$, la política codiciosa derivada de $Q$ coincidirá con la política óptima de Value Iteration en al menos el 90 por ciento de los 144 estados, y al menos el 80 por ciento de los desacuerdos estará en estados con menos de 100 visitas acumuladas.

Cómo se falsa: se calcula el porcentaje de estados donde $\arg\max_a Q_{MC}(s, a)$ difiere de $\pi^*_{VI}(s)$ y se cruza con el contador de visitas por estado. Si la coincidencia queda por debajo del 90 por ciento, o si los desacuerdos aparecen en estados muy visitados, la hipótesis falla y el problema no es falta de datos sino algo estructural del método.

Esperamos además que los desacuerdos se concentren en la decisión de desviarse hacia T2, porque las dos mejores rutas del Inciso 1 se diferencian en solo 0.09 de retorno y el ruido de Monte Carlo puede invertir ese orden.

**Hipótesis 2: precisión del valor y costo en muestras**

El valor del estado inicial estimado por Monte Carlo se acercará al de Value Iteration a ritmo $1/\sqrt{N}$. En concreto, $|V_{MC}(s_0) - V_{VI}(s_0)| > 0.5$ todavía a los 1000 episodios y $< 0.1$ a los 20000 episodios, mientras que Value Iteration alcanzará un error menor a $10^{-6}$ en menos de 400 barridos. Por diseño esperamos $V_{VI}(s_0) \approx 8.65$, que corresponde a la ruta que recoge T1 y sigue a la salida.

Cómo se falsa: se grafica el error contra el número de episodios en escala logarítmica. Si la pendiente no se parece a $-1/2$, o si el error baja de 0.1 mucho antes de los 20000 episodios, la hipótesis falla. También falla si $V_{VI}(s_0)$ resulta distinto de 8.65, lo que indicaría que el análisis de rutas del Inciso 1 dejó fuera alguna ruta mejor.

El umbral de 0.1 no es arbitrario: para que Monte Carlo elija la misma ruta que Value Iteration necesita resolver una diferencia de 0.09 entre las dos mejores rutas, así que cualquier error por encima de eso deja la decisión en manos del azar.

**Hipótesis 3: reproducibilidad**

Con 5 semillas independientes y 10000 episodios cada una, la desviación estándar de $V_{MC}(s_0)$ entre semillas será mayor a 0.2, mientras que Value Iteration devolverá exactamente el mismo valor en las 5 corridas porque no muestrea nada.

Cómo se falsa: se corren las 5 semillas y se mide la dispersión. Si la desviación entre semillas es menor a 0.2, significa que la varianza del retorno en este mapa es mucho menor de lo que anticipamos y que Monte Carlo es más barato de lo previsto para este dominio.

# Task 2 (Entrega Parcial)

Respondan las siguientes preguntas con argumentación técnica.

## Inciso 1

Para el MDP que diseñaron, calculen una cota superior del número de episodios necesarios para que todos los pares $(s, a)$ sean visitados al menos una vez en esperanza, bajo una política uniforme aleatoria. Expresen el resultado en función de $|S|$ y $|A|$ y evalúen numéricamente para su MDP específico.

**Respuesta:**

**Planteamiento**

Sea $p_{s,a}$ la probabilidad de que el par $(s, a)$ aparezca al menos una vez en un episodio. Los episodios son independientes entre sí, así que el número de episodios hasta que aparece un par fijo es geométrico de media $1/p_{s,a}$. Lo que se pide es el número de episodios hasta que aparecen todos los pares, que es el problema del coleccionista de cupones con probabilidades desiguales. Usando $p = \min_{s,a} p_{s,a}$ se obtiene la cota

$$\mathbb{E}[N] \le \frac{1}{p} H_{|S||A|}, \qquad H_m = \sum_{k=1}^{m} \frac{1}{k} \approx \ln m + 0.5772$$

El factor $H_m$ es el precio de tener que esperar al último par que falta y no solo a uno cualquiera.

**Cota de $p$**

Bajo política uniforme, $\pi(a \mid s) = 1/|A|$ en todo estado, así que

$$p_{s,a} \ge \frac{p_s}{|A|}$$

donde $p_s$ es la probabilidad de visitar el estado $s$ durante el episodio. Si el episodio es lo bastante largo como para que la caminata aleatoria recorra la cuadrícula, todos los estados se visitan y el menos favorecido se lleva al menos la parte uniforme, $p_s \ge 1/|S|$. Con eso, $p \ge 1/(|S||A|)$ y la cota queda

$$\mathbb{E}[N] \le |S|\,|A|\,\big(\ln(|S||A|) + 0.5772\big)$$

**Evaluación numérica**

Para el MDP diseñado, $|S| = 144$ y $|A| = 4$, de modo que hay $|S||A| = 576$ pares:

$$\mathbb{E}[N] \le 576 \cdot (\ln 576 + 0.5772) = 576 \cdot (6.356 + 0.577) = 576 \cdot 6.933 \approx 3994$$

Es decir, del orden de $4.0 \times 10^3$ episodios.

Si se ignoran los bits de cofre recogido y se cuentan solo las 36 celdas, quedan 144 pares y la cota baja a $144 \cdot 5.55 \approx 799$ episodios. La diferencia entre 800 y 4000 es el precio que se paga por haber ampliado el estado con los dos bits, y es un precio razonable comparado con la alternativa de que el episodio nunca termine.

**Supuestos y qué pasa si no se cumplen**

1. El episodio debe durar más que el tiempo de cobertura de la cuadrícula. Nuestro corte de 200 pasos está en el mismo orden que ese tiempo de cobertura, no claramente por encima, así que en la práctica hay que esperar un número mayor al de la cota, o subir el corte.
2. Se asume que todos los pares son alcanzables. Los estados con $b_1 = 1$ solo existen después de pasar por T1, así que su $p_s$ real es menor que $1/|S|$ y la cota los subestima.
3. Es una cota superior floja a propósito. Sirve para dimensionar el experimento, no para predecir el valor exacto, y dice lo importante: el orden de magnitud es miles de episodios solo para tocar cada par una vez, muy lejos de la sola visita por par que necesitaría Value Iteration.

Como referencia, con Exploring Starts uniformes sobre los pares el problema se vuelve el coleccionista de cupones clásico y da $|S||A| H_{|S||A|} \approx 3994$, el mismo número. Eso confirma que la cota está construida para ser comparable con el mejor caso posible de exploración.

## Inciso 2

El retorno $G_t$ es un estimador insesgado pero de alta varianza de $V^\pi(s)$. Expliquen formalmente de dónde proviene esa varianza, por qué crece con la longitud del episodio, y qué consecuencia tiene sobre el número de episodios necesarios para convergencia práctica.

**Respuesta:**

**De dónde viene la varianza**

El retorno es una suma de variables aleatorias:

$$G_t = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$

y cada una de sus fuentes de aleatoriedad se acumula en la suma:

1. La selección de acciones. Bajo $\varepsilon$-soft cada paso es una moneda cargada, y una sola desviación temprana puede mandar al personaje por un lado del mapa completamente distinto.
2. Las transiciones del entorno. En nuestro diseño son deterministas, pero en el caso general cada acción abre un abanico de sucesores.
3. La longitud $T$ del episodio, que es en sí misma aleatoria porque depende de cuánto tarde el personaje en llegar a la salida.
4. Qué recompensas se cobran. Dos trayectorias desde el mismo estado pueden diferir en si recogieron T1, si cayeron en la trampa, o cuántos pasos gastaron, y eso cambia el retorno en decenas de unidades.

Formalmente, la varianza de una suma no es solo la suma de varianzas:

$$\text{Var}(G_t) = \sum_k \gamma^{2k}\,\text{Var}(R_{t+k+1}) \; + \; 2\sum_{i<j} \gamma^{i+j}\,\text{Cov}(R_{t+i+1}, R_{t+j+1})$$

**Por qué crece con la longitud del episodio**

En la fórmula anterior hay $T$ términos de varianza y $T(T-1)/2$ términos de covarianza. Con $\gamma = 1$ eso significa que la varianza crece como $T$ cuando las recompensas están poco correlacionadas y hasta como $T^2$ cuando están fuertemente correlacionadas, que es el caso típico porque los pasos de una misma trayectoria comparten historia. Cada paso adicional agrega una recompensa aleatoria más y correlaciones con todas las anteriores.

Con $\gamma < 1$ el crecimiento se frena porque los términos lejanos pesan poco:

$$\sum_{k=0}^{\infty} \gamma^{2k}\sigma^2 = \frac{\sigma^2}{1-\gamma^2}, \qquad |G_t| \le \frac{R_{\max}}{1-\gamma}$$

Para nuestro MDP, con $R_{\max} = 20$ y $\gamma = 0.95$, el retorno vive en un rango de hasta $\pm 400$ mientras que los valores reales rondan entre 4 y 9. La cota es floja, pero muestra el punto: la varianza crece con el mínimo entre la longitud del episodio y el horizonte efectivo $1/(1-\gamma) = 20$. Bajo política aleatoria los episodios duran cientos de pasos, así que el horizonte efectivo es lo que manda y la varianza se satura en un valor alto.

**Consecuencia sobre el número de episodios**

La estimación Monte Carlo de $V^\pi(s)$ es el promedio de $n$ retornos independientes, así que su error estándar es $\sigma_G/\sqrt{n}$. Para garantizar un error menor a $\epsilon$ con 95 por ciento de confianza hace falta

$$n \ge \left(\frac{1.96\,\sigma_G}{\epsilon}\right)^2$$

La relación es cuadrática tanto en $1/\epsilon$ como en $\sigma_G$. Bajar el error a la mitad cuesta cuatro veces más episodios, y duplicar la desviación del retorno también cuesta cuatro veces más. Con un $\sigma_G$ moderado de alrededor de 6 unidades, llegar a $\epsilon = 0.1$ pide unas 13800 muestras para un solo estado, y eso multiplicado por los estados que se visitan poco.

Aquí está la diferencia de fondo con Value Iteration: Programación Dinámica no muestrea nada, así que no tiene varianza y su error baja geométricamente con el número de barridos. Monte Carlo paga con muestras el no necesitar el modelo, y esa cuenta es la que decide si el método es viable para el estudio o no.

## Inciso 3

Argumenten formalmente por qué Monte Carlo no puede aplicarse directamente a tareas continuas. Propongan una modificación concreta al algoritmo que permita aproximar Monte Carlo en una tarea continua, y discutan las implicaciones de esa modificación sobre las garantías de convergencia.

**Respuesta:**

**Por qué no se puede aplicar directamente**

Monte Carlo actualiza $V(s)$ con el promedio de los retornos observados, y el retorno es

$$G_t = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$

Esta expresión depende de $T$, el instante de terminación. En una tarea continua no existe estado terminal, de modo que $T = \infty$ y la suma tiene infinitos términos. El problema no es de convergencia matemática: para $\gamma < 1$ la esperanza $V^\pi(s) = \mathbb{E}[G_t]$ existe y es finita porque la serie está acotada por $R_{\max}/(1-\gamma)$. El problema es de observabilidad. Monte Carlo tiene que esperar a que el episodio termine para hacer la primera actualización, y ese final nunca llega, así que la muestra de $G_t$ no está disponible en ningún tiempo finito y el algoritmo simplemente nunca actualiza nada.

Con $\gamma = 1$ se suma un segundo problema, y este sí es matemático: el retorno diverge y ni siquiera existe la cantidad que se quiere estimar.

**Modificación propuesta: retorno truncado con pseudo episodios**

Se corta el retorno en un horizonte fijo $H$ y se tratan bloques de $H$ pasos como si fueran episodios:

$$G_t^{(H)} = \sum_{k=0}^{H-1} \gamma^k R_{t+k+1}$$

El pedazo que se ignora está acotado por la cola de la serie geométrica:

$$\big|G_t - G_t^{(H)}\big| \le \frac{\gamma^H R_{\max}}{1-\gamma}$$

lo que permite elegir $H$ a partir de la tolerancia deseada. Con $\gamma = 0.95$, $R_{\max} = 20$ y una tolerancia de 0.1:

$$0.95^H \cdot 400 \le 0.1 \;\implies\; H \ge \frac{\ln(2.5 \times 10^{-4})}{\ln 0.95} = 161.7 \;\implies\; H = 162$$

**Implicaciones sobre las garantías**

1. Se pierde el insesgamiento. El estimador ya no converge a $V^\pi(s)$ sino a un punto que difiere de él en a lo sumo $\gamma^H R_{\max}/(1-\gamma)$. La garantía pasa de converger al valor verdadero a converger a un entorno de radio conocido. Es un sesgo controlado, porque se puede hacer tan pequeño como se quiera subiendo $H$, pero deja de ser cero.
2. Aparece un intercambio entre sesgo y varianza. Subir $H$ reduce el sesgo geométricamente pero agrega términos a la suma, sube la varianza del retorno y por lo tanto exige más pseudo episodios para la misma precisión. Bajar $H$ hace lo contrario. Ese balance no existía en la versión episódica.
3. Se rompe la independencia de las muestras. Los pseudo episodios son pedazos contiguos de una misma trayectoria, así que su estado inicial no es un sorteo independiente y el argumento clásico de la ley de los grandes números sobre retornos independientes deja de aplicar tal cual. Todavía se puede argumentar convergencia si la cadena es ergódica y visita cada estado infinitas veces, pero la garantía se debilita: la velocidad ahora depende del tiempo de mezcla de la cadena y no solo del número de muestras.
4. Se necesita $\gamma < 1$ estricto. La modificación descansa por completo en que la cola geométrica sea despreciable, así que en una tarea continua con recompensa promedio y $\gamma = 1$ hay que cambiar de formulación, por ejemplo a la de recompensa promedio con valores diferenciales, y eso ya no es Monte Carlo truncado.

Una variante del mismo remedio es no descartar la cola sino estimarla, sumando $\gamma^H V(s_{t+H})$ al retorno truncado. Eso reduce mucho el sesgo y la varianza a la vez, pero introduce sesgo de bootstrap y ya no es Monte Carlo puro sino TD de $n$ pasos, que es justo el método que aparece cuando se admite que Monte Carlo no encaja bien en tareas que no terminan.